# Stormlight chapter stylometry — PCA by POV

Each dot is one section (chapter, prologue, interlude, or epilogue) from *The Way of Kings* or *Words of Radiance*.
Position is the first two principal components of relative word-frequency vectors built from the section's text.
Color encodes the POV character, marker shape encodes the book.

Tunable knobs are in the config cell: vocabulary size, min-word filter, and how many POVs get distinct colors before the rest collapse into a grey "Other" bucket.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from scipy.spatial import ConvexHull
from pathlib import Path

# ── Config ───────────────────────────────────────────────────────────────────
REPO            = Path("/Users/caputomachine/Desktop/StormlightCorpus")
CSV_PATH        = REPO / "csv_data" / "chapters_full.csv"
STOPWORDS_FILE  = REPO / "stormlight_viz" / "stormlight_stopwords.txt"
OUT_PNG         = "pov_pca_stormlight.png"
OUT_HTML        = "pov_pca_interactive.html"

MIN_WORDS           = 500    # drop fragments shorter than this
MAX_FEATURES        = 1000   # DTM vocabulary size (bigrams need extra room)
MIN_DF              = 2      # word/bigram must appear in ≥2 chapters
MAX_DF              = 0.95   # drop tokens appearing in >95% of chapters
NGRAM_RANGE         = (1, 2) # unigrams + bigrams
TOP_K_POVS_COLORED  = 8      # top-K POVs get distinct colors; rest are grey
LOWERCASE           = True

# Character/place name sections to strip from the DTM (NOT the literary-noise
# sections — dialogue tags like "said/looked/thought" are stylistic markers and
# should remain as features for stylometry).
CHARACTER_SECTIONS = {
    "POV characters",
    "Major non-POV characters",
    "Bridge Four",
    "Alethi highprinces",
    "Other Heralds",
    "Places",
    "Peoples",
}


In [ ]:
# ── 1. Load + filter ─────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
df = df[df["section_type"] != "part"]
df = df[df["word_count"] >= MIN_WORDS]
df["pov"] = df["pov"].fillna("Unknown")
# Sort by narrative order so trajectory lines later connect chapters chronologically
df = df.sort_values(["book", "order"]).reset_index(drop=True)

print(f"sections : {len(df)}")
print(f"books    : {df['book'].nunique()}  ({sorted(df['book'].unique())})")
print(f"POVs     : {df['pov'].nunique()}")
print()
print("Top POVs by section count:")
print(df["pov"].value_counts().head(12).to_string())


In [ ]:
# ── 2. Vectorize + PCA ───────────────────────────────────────────────────────
# Build stopword list: English + Stormlight character/place names
def load_name_stopwords(path: Path, sections: set) -> set:
    in_section = False
    words = set()
    with open(path) as f:
        for line in f:
            stripped = line.strip()
            if stripped.startswith("#"):
                comment = stripped.lstrip("#").strip(" ─-")
                in_section = any(s in comment for s in sections)
                continue
            if not stripped:
                continue
            if in_section:
                words.add(stripped.lower())
    return words

name_sw = load_name_stopwords(STOPWORDS_FILE, CHARACTER_SECTIONS)
all_sw  = sorted(set(ENGLISH_STOP_WORDS) | name_sw)
print(f"stopwords: {len(name_sw)} character/place names + {len(ENGLISH_STOP_WORDS)} English = {len(all_sw)} total")

vectorizer = CountVectorizer(
    lowercase=LOWERCASE,
    stop_words=all_sw,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
)
dtm = vectorizer.fit_transform(df["text"].tolist())
print(f"DTM shape: {dtm.shape}")

# Relative frequencies, so long chapters don't dominate
dtm_rel = normalize(dtm, norm="l1", axis=1).toarray()

# ── Confound control: subtract per-book mean ─────────────────────────────────
# Removes the global "WoK style vs WoR style" effect so PC1 reflects POV style
# rather than which book the chapter belongs to.
dtm_centered = dtm_rel.copy()
for book in df["book"].unique():
    mask = (df["book"] == book).values
    book_mean = dtm_centered[mask].mean(axis=0)
    dtm_centered[mask] -= book_mean
print(f"per-book mean subtracted; mean magnitude after: {np.abs(dtm_centered).mean():.5f}")

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(dtm_centered)
var = pca.explained_variance_ratio_

print(f"PC1 variance    : {var[0]:.1%}")
print(f"PC2 variance    : {var[1]:.1%}")
print(f"Cumulative 1+2  : {var.sum():.1%}")

# Inspect: top features driving each PC
feat_names = vectorizer.get_feature_names_out()
def top_features(component, n=10):
    idx_pos = np.argsort(component)[-n:][::-1]
    idx_neg = np.argsort(component)[:n]
    return ([(feat_names[i], component[i]) for i in idx_pos],
            [(feat_names[i], component[i]) for i in idx_neg])

pc1_pos, pc1_neg = top_features(pca.components_[0])
pc2_pos, pc2_neg = top_features(pca.components_[1])
print(f"\nPC1 positive: {', '.join(w for w,_ in pc1_pos[:8])}")
print(f"PC1 negative: {', '.join(w for w,_ in pc1_neg[:8])}")
print(f"PC2 positive: {', '.join(w for w,_ in pc2_pos[:8])}")
print(f"PC2 negative: {', '.join(w for w,_ in pc2_neg[:8])}")


In [ ]:
# ── 3. Color (POV) + marker (book) ───────────────────────────────────────────
pov_counts = df["pov"].value_counts()
top_povs = list(pov_counts.head(TOP_K_POVS_COLORED).index)

palette = plt.get_cmap("tab10", len(top_povs))
pov_color = {p: palette(i) for i, p in enumerate(top_povs)}
OTHER_COLOR = (0.72, 0.72, 0.74, 0.8)

def color_for(pov):
    return pov_color.get(pov, OTHER_COLOR)

book_marker = {"The Way of Kings": "o", "Words of Radiance": "s"}


In [ ]:
# ── 4. Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 9))

# Layer 1: convex hulls per top POV (need ≥3 points)
for pov in top_povs:
    idx = df.index[df["pov"] == pov].tolist()
    pts = coords[idx]
    if len(pts) < 3:
        continue
    hull = ConvexHull(pts)
    hull_pts = pts[hull.vertices]
    ax.fill(hull_pts[:, 0], hull_pts[:, 1],
            color=pov_color[pov], alpha=0.10, zorder=1)
    ax.plot(np.append(hull_pts[:, 0], hull_pts[0, 0]),
            np.append(hull_pts[:, 1], hull_pts[0, 1]),
            color=pov_color[pov], alpha=0.35, linewidth=1.0, zorder=1)

# Layer 3: scatter points
for book, marker in book_marker.items():
    for pov in df["pov"].unique():
        mask = (df["book"] == book) & (df["pov"] == pov)
        if not mask.any():
            continue
        idx = np.where(mask.values)[0]
        ax.scatter(coords[idx, 0], coords[idx, 1],
                   color=color_for(pov), marker=marker, s=70,
                   edgecolors="white", linewidths=0.7, alpha=0.92, zorder=3)

# Legend 1: POVs
pov_handles = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=pov_color[p], markeredgecolor="white",
               markersize=10, label=f"{p}  ({pov_counts[p]})")
    for p in top_povs
]
n_other = int((~df["pov"].isin(top_povs)).sum())
if n_other:
    pov_handles.append(
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=OTHER_COLOR, markeredgecolor="white",
                   markersize=10, label=f"Other  ({n_other})")
    )
leg1 = ax.legend(handles=pov_handles, title="POV (n sections)",
                 bbox_to_anchor=(1.02, 1), loc="upper left",
                 borderaxespad=0, frameon=False)
ax.add_artist(leg1)

# Legend 2: books
book_handles = [
    plt.Line2D([0], [0], marker=m, color="black", linestyle="None",
               markerfacecolor="lightgray", markeredgecolor="black",
               markersize=10, label=b)
    for b, m in book_marker.items()
]
ax.legend(handles=book_handles, title="Book",
          bbox_to_anchor=(1.02, 0.30), loc="upper left",
          borderaxespad=0, frameon=False)

ax.set_xlabel(f"PC1 ({var[0]:.1%} variance)", fontsize=11)
ax.set_ylabel(f"PC2 ({var[1]:.1%} variance)", fontsize=11)
ax.set_title(
    f"Stormlight chapters — PCA of relative word frequencies\n"
    f"unigrams + bigrams, character names stripped, per-book mean removed   |   n = {len(df)} sections",
    fontsize=12, fontweight="bold",
)
ax.axhline(0, color="lightgrey", linewidth=0.8, zorder=0)
ax.axvline(0, color="lightgrey", linewidth=0.8, zorder=0)
ax.grid(True, linestyle="--", alpha=0.30, zorder=0)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {OUT_PNG}")


## Interactive version

Same PCA, rendered with Plotly so you can hover any dot to see the chapter title, POV, book, and word count. Also writes a self-contained HTML file you can open outside Jupyter.

Requires `plotly` (`pip install plotly`).


In [ ]:
# ── 5. Interactive plot — hover for POV + chapter title ──────────────────────
# Centroids hidden by default; pick a POV from the dropdown (upper-right) to
# isolate that character's chapters + hull and reveal their centroid.
import plotly.graph_objects as go
import plotly.colors as pc

plot_df = df.copy()
plot_df["PC1"] = coords[:, 0]
plot_df["PC2"] = coords[:, 1]
plot_df["pov_group"] = plot_df["pov"].where(plot_df["pov"].isin(top_povs), "Other")

qual = pc.qualitative.D3
color_map_hex = {p: qual[i % len(qual)] for i, p in enumerate(top_povs)}
color_map_hex["Other"] = "#bfbfbf"

fig = go.Figure()
trace_meta: list[tuple[str, str]] = []   # (category, pov_group) per trace

# ── Layer 1: convex hulls
for pov in top_povs:
    pts = coords[(plot_df["pov"] == pov).values]
    if len(pts) < 3:
        continue
    hull = ConvexHull(pts)
    hpts = pts[hull.vertices]
    hpts = np.vstack([hpts, hpts[0:1]])
    fig.add_trace(go.Scatter(
        x=hpts[:, 0], y=hpts[:, 1],
        fill="toself", fillcolor=color_map_hex[pov],
        opacity=0.12, line=dict(color=color_map_hex[pov], width=1.2),
        name=f"{pov} hull", legendgroup=pov, showlegend=False,
        hoverinfo="skip",
    ))
    trace_meta.append(("hull", pov))

# ── Layer 2: scatter points (with hover)
for book, marker in [("The Way of Kings", "circle"), ("Words of Radiance", "square")]:
    for pov in plot_df["pov"].unique():
        sub = plot_df[(plot_df["book"] == book) & (plot_df["pov"] == pov)]
        if sub.empty:
            continue
        pov_grp = pov if pov in top_povs else "Other"
        fig.add_trace(go.Scatter(
            x=sub["PC1"], y=sub["PC2"], mode="markers",
            marker=dict(size=10, symbol=marker, color=color_map_hex[pov_grp],
                        line=dict(color="white", width=0.8)),
            name=f"{pov} ({book.split()[-1]})",
            legendgroup=pov_grp,
            customdata=np.stack([sub["heading_text"], sub["pov"], sub["book"],
                                 sub["word_count"]], axis=-1),
            hovertemplate="<b>%{customdata[0]}</b><br>"
                          "POV: %{customdata[1]}<br>"
                          "Book: %{customdata[2]}<br>"
                          "Words: %{customdata[3]}<br>"
                          "PC1: %{x:.4f}  PC2: %{y:.4f}<extra></extra>",
            showlegend=False,
        ))
        trace_meta.append(("scatter", pov_grp))

# ── Layer 3: centroids (one trace per POV, hidden until selected)
for pov in top_povs:
    sub = plot_df[plot_df["pov"] == pov]
    if len(sub) < 2:
        continue
    fig.add_trace(go.Scatter(
        x=[sub["PC1"].mean()], y=[sub["PC2"].mean()],
        mode="markers+text", text=[pov],
        textposition="top center",
        textfont=dict(size=13, color="black"),
        marker=dict(size=20, symbol="x-thin", color=color_map_hex[pov],
                    line=dict(color="black", width=2.5)),
        name=f"{pov} centroid",
        visible=False, hoverinfo="skip", showlegend=False,
    ))
    trace_meta.append(("centroid", pov))

# ── Legend stand-ins (always visible across dropdown states)
for pov in top_povs:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=11, color=color_map_hex[pov],
                    line=dict(color="white", width=0.8)),
        name=f"{pov}  ({pov_counts[pov]})", legendgroup=pov, showlegend=True,
    ))
    trace_meta.append(("legend", pov))
if n_other:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=11, color=color_map_hex["Other"],
                    line=dict(color="white", width=0.8)),
        name=f"Other  ({n_other})", showlegend=True,
    ))
    trace_meta.append(("legend", "Other"))

# ── Build dropdown buttons
def visibility_for(selected_pov):
    out = []
    for cat, pov_tag in trace_meta:
        if cat == "legend":
            out.append(True)
        elif cat == "centroid":
            out.append(selected_pov is not None and pov_tag == selected_pov)
        else:  # hull / scatter
            if selected_pov is None:
                out.append(True)
            else:
                out.append(pov_tag == selected_pov)
    return out

buttons = [dict(label="All POVs", method="update",
                args=[{"visible": visibility_for(None)}])]
for pov in top_povs:
    buttons.append(dict(label=pov, method="update",
                        args=[{"visible": visibility_for(pov)}]))

# Apply the initial (All POVs) visibility so first render matches the dropdown's default
for i, v in enumerate(visibility_for(None)):
    fig.data[i].visible = v

fig.update_layout(
    width=1150, height=780,
    title=(f"Stormlight chapters — PCA of relative word frequencies "
           f"(unigrams + bigrams, names stripped, per-book mean removed)  |  n = {len(df)} sections"),
    xaxis=dict(title=f"PC1 ({var[0]:.1%} variance)", zeroline=True,
               zerolinecolor="lightgrey", gridcolor="whitesmoke"),
    yaxis=dict(title=f"PC2 ({var[1]:.1%} variance)", zeroline=True,
               zerolinecolor="lightgrey", gridcolor="whitesmoke"),
    plot_bgcolor="white",
    legend=dict(title="POV (n sections)", font=dict(size=10)),
    hovermode="closest",
    updatemenus=[dict(
        buttons=buttons, direction="down",
        x=1.22, y=0.02, xanchor="left", yanchor="bottom",
        showactive=True, type="dropdown",
        bgcolor="white", bordercolor="lightgrey", borderwidth=1,
        font=dict(size=11), pad=dict(t=4, b=4, l=8, r=8),
    )],
    annotations=[dict(
        text="Highlight POV:", showarrow=False,
        x=1.22, y=0.10, xref="paper", yref="paper",
        xanchor="left", yanchor="bottom",
        font=dict(size=11, color="#444"),
    )],
)

fig.write_html(OUT_HTML, include_plotlyjs="cdn")
print(f"saved {OUT_HTML}  (use the dropdown at lower-right to isolate a POV)")
fig.show()


## Export for the Dash app

Writes two artifacts the Dash app consumes:

- `csv_data/chapters_pca.csv` — one row per chapter with `pc1`, `pc2`, and full metadata
- `csv_data/pca_meta.json` — per-POV centroids + convex hulls, PC variances, top loadings, vocab config

No scipy needed at runtime — hulls and centroids are pre-computed here. The Dash figure module just reads these files and rebuilds the Plotly figure with the user's filter applied.


In [ ]:
# ── 6. Export data for the Dash app ──────────────────────────────────────────
import json

PCA_CSV  = REPO / "csv_data" / "chapters_pca.csv"
PCA_META = REPO / "csv_data" / "pca_meta.json"

out_df = df.copy()
out_df["pc1"] = coords[:, 0]
out_df["pc2"] = coords[:, 1]
out_df = out_df[["book", "order", "heading_id", "section_type", "title",
                 "heading_text", "pov", "word_count", "pc1", "pc2"]]
out_df.to_csv(PCA_CSV, index=False)

overlays = {}
for pov in top_povs:
    sub = out_df[out_df["pov"] == pov]
    if len(sub) < 2:
        continue
    pts = sub[["pc1", "pc2"]].values
    entry = {
        "centroid": [float(pts[:, 0].mean()), float(pts[:, 1].mean())],
        "n_sections": int(len(sub)),
    }
    if len(pts) >= 3:
        hull = ConvexHull(pts)
        entry["hull"] = [[float(pts[i, 0]), float(pts[i, 1])] for i in hull.vertices]
    overlays[pov] = entry

# Top-loading words (for the Dash tab description / tooltip later)
feat_names = vectorizer.get_feature_names_out()
def top_features_list(component, n=10):
    pos = np.argsort(component)[-n:][::-1]
    neg = np.argsort(component)[:n]
    return ([feat_names[i] for i in pos], [feat_names[i] for i in neg])

pc1_pos, pc1_neg = top_features_list(pca.components_[0])
pc2_pos, pc2_neg = top_features_list(pca.components_[1])

meta = {
    "pc1_variance": float(var[0]),
    "pc2_variance": float(var[1]),
    "pc1_loadings_pos": pc1_pos,
    "pc1_loadings_neg": pc1_neg,
    "pc2_loadings_pos": pc2_pos,
    "pc2_loadings_neg": pc2_neg,
    "n_sections": int(len(df)),
    "top_povs": top_povs,
    "pov_counts": {p: int(pov_counts[p]) for p in pov_counts.index},
    "overlays": overlays,
    "config": {
        "min_words": MIN_WORDS,
        "max_features": MAX_FEATURES,
        "ngram_range": list(NGRAM_RANGE),
        "min_df": MIN_DF,
        "max_df": MAX_DF,
        "lowercase": LOWERCASE,
        "book_mean_subtracted": True,
        "name_stopwords_used": True,
    },
}
with open(PCA_META, "w") as f:
    json.dump(meta, f, indent=2)

print(f"wrote {PCA_CSV.relative_to(REPO)}  ({len(out_df)} rows)")
print(f"wrote {PCA_META.relative_to(REPO)}  ({len(overlays)} POV overlays)")
